# NanoAOD Structure

This notebook documents the NanoAOD branch structure as used by the DarkBottomLine analysis framework.

## Overview

CMS NanoAOD is a flat-ntuple format where physics objects are stored as **arrays of branches** with a common prefix:
- `Muon_*` — muon collection
- `Electron_*` — electron collection
- `Jet_*` — AK4 jet collection
- `FatJet_*` — AK8 fat jet collection
- `Tau_*` — tau collection
- `MET_*` / `PFMET_*` — missing transverse energy
- `Pileup_*` — pileup information
- `event`, `run`, `luminosityBlock` — event metadata

This analysis uses **NanoAODv12** (2022 data) and **NanoAODv15** (2024–2025 data).
The key difference is the MET branch naming: `PFMET_*` is prioritised in v15 if available.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import pandas as pd

## Branches read by this analysis

The table below lists every NanoAOD branch explicitly accessed in `darkbottomline/objects.py`
and `darkbottomline/processor.py`, grouped by collection.

In [ ]:
# Branches accessed by this analysis, derived from objects.py and processor.py
branch_table = [
    # ── Muons ──────────────────────────────────────────────────────────────
    ("Muon",     "Muon_pt",                    "float",  "Transverse momentum [GeV]"),
    ("Muon",     "Muon_eta",                   "float",  "Pseudorapidity"),
    ("Muon",     "Muon_phi",                   "float",  "Azimuthal angle [rad]"),
    ("Muon",     "Muon_tightId",               "bool",   "Tight muon ID flag"),
    ("Muon",     "Muon_looseId",               "bool",   "Loose muon ID flag (optional, v12+)"),
    ("Muon",     "Muon_pfIsoId",               "int",    "PF isolation working point (1=loose … 4=tight)"),
    ("Muon",     "Muon_charge",                "int",    "Electric charge (±1), used for Z→μμ pairing"),
    ("Muon",     "Muon_mass",                  "float",  "PDG mass [GeV] (optional, for dilepton mass)"),
    # ── Electrons ──────────────────────────────────────────────────────────
    ("Electron", "Electron_pt",                "float",  "Transverse momentum [GeV]"),
    ("Electron", "Electron_eta",               "float",  "Pseudorapidity (supercluster)"),
    ("Electron", "Electron_phi",               "float",  "Azimuthal angle [rad]"),
    ("Electron", "Electron_cutBased",          "int",    "Cut-based ID (0=fail, 1=veto, 2=loose, 3=medium, 4=tight)"),
    ("Electron", "Electron_mvaIso_WP80",       "bool",   "MVA ISO WP80 (tight)"),
    ("Electron", "Electron_mvaIso_WP90",       "bool",   "MVA ISO WP90 (loose, optional)"),
    ("Electron", "Electron_charge",            "int",    "Electric charge (±1), used for Z→ee pairing"),
    ("Electron", "Electron_mass",              "float",  "Mass [GeV] (optional, for dilepton mass)"),
    # ── Taus ───────────────────────────────────────────────────────────────
    ("Tau",      "Tau_pt",                     "float",  "Transverse momentum [GeV]"),
    ("Tau",      "Tau_eta",                    "float",  "Pseudorapidity"),
    ("Tau",      "Tau_phi",                    "float",  "Azimuthal angle [rad]"),
    ("Tau",      "Tau_idDeepTau2018v2p5VSjet", "int",    "DeepTau v2p5 vs-jet ID (4=VLoose, 8=Loose, 16=Medium, 32=Tight …)"),
    ("Tau",      "Tau_decayMode",              "int",    "Decay mode (allowed: 0,1,2,10,11)"),
    # ── AK4 Jets ───────────────────────────────────────────────────────────
    ("Jet",      "Jet_pt",                     "float",  "Transverse momentum (JEC applied) [GeV]"),
    ("Jet",      "Jet_eta",                    "float",  "Pseudorapidity"),
    ("Jet",      "Jet_phi",                    "float",  "Azimuthal angle [rad]"),
    ("Jet",      "Jet_btagDeepFlavB",          "float",  "DeepJet b-tagging discriminant (0–1)"),
    ("Jet",      "Jet_hadronFlavour",          "int",    "Gen-level hadron flavour (5=b, 4=c, 0=udsg); MC only"),
    ("Jet",      "Jet_partonFlavour",          "int",    "Parton flavour (fallback if hadronFlavour absent)"),
    # ── AK8 Fat Jets ───────────────────────────────────────────────────────
    ("FatJet",   "FatJet_pt",                  "float",  "Transverse momentum [GeV]"),
    ("FatJet",   "FatJet_eta",                 "float",  "Pseudorapidity"),
    ("FatJet",   "FatJet_phi",                 "float",  "Azimuthal angle [rad]"),
    ("FatJet",   "FatJet_mass",                "float",  "Soft-drop mass [GeV]"),
    # ── MET ────────────────────────────────────────────────────────────────
    ("MET",      "MET_pt",                     "float",  "Missing transverse energy magnitude [GeV]"),
    ("MET",      "MET_phi",                    "float",  "MET azimuthal angle [rad]"),
    ("MET",      "MET_significance",           "float",  "MET significance"),
    ("MET",      "PFMET_pt",                   "float",  "PF MET [GeV] (v15 priority over MET_pt)"),
    ("MET",      "PFMET_phi",                  "float",  "PF MET phi [rad] (v15)"),
    ("MET",      "PFMET_significance",         "float",  "PF MET significance (v15)"),
    # ── Pileup & Event ─────────────────────────────────────────────────────
    ("Pileup",   "Pileup_nTrueInt",            "float",  "True number of pileup interactions (MC only)"),
    ("Event",    "event",                      "uint64", "Event number"),
    ("Event",    "run",                        "uint32", "Run number"),
    ("Event",    "luminosityBlock",            "uint32", "Luminosity block number"),
]

df = pd.DataFrame(branch_table, columns=["Collection", "Branch", "Type", "Description"])
df = df.set_index(["Collection", "Branch"])
print(f"Total branches used: {len(df)}")
df

## Object selection thresholds (from config)

Each collection is selected with the following pT / η / ID thresholds,
loaded from the year-specific config file (e.g. `configs/2022.yaml`).

In [ ]:
import yaml

# Load a representative year config (2022) to show object thresholds
config_path = project_root / "configs" / "2022.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f)

objects = cfg.get("objects", {})

selection_rows = []
for obj_name, obj_cfg in objects.items():
    row = {
        "Object": obj_name,
        "pT min (tight) [GeV]": obj_cfg.get("pt_min", "–"),
        "pT min (loose) [GeV]": obj_cfg.get("pt_min_loose", "–"),
        "|η| max": obj_cfg.get("eta_max", "–"),
        "Extra cuts": ", ".join(
            f"{k}={v}" for k, v in obj_cfg.items()
            if k not in ("pt_min", "pt_min_loose", "eta_max")
        ) or "–",
    }
    selection_rows.append(row)

sel_df = pd.DataFrame(selection_rows).set_index("Object")
print(f"Config: {config_path.name}  (year {cfg.get('year', '?')})")
sel_df

## Inspecting a NanoAOD file with uproot

If you have a NanoAOD ROOT file available, the cells below show how to
open it, list its branches, and load a collection into an awkward array.

> Set `NANOAOD_PATH` to point to a local or XRootD-accessible file.

In [ ]:
import uproot

# ── Set this to a real NanoAOD file path (local or xrootd) ────────────────
NANOAOD_PATH = None  # e.g. "/path/to/NanoAODv12.root"  or  "root://cms-xrd-global.cern.ch//store/..."
# ─────────────────────────────────────────────────────────────────────────

if NANOAOD_PATH is None:
    print("NANOAOD_PATH not set — skipping file open. Set it above to inspect a real file.")
else:
    f = uproot.open(NANOAOD_PATH)
    tree = f["Events"]
    print(f"Tree: {tree}")
    print(f"Entries: {tree.num_entries:,}")
    print(f"\nAll branch names ({len(tree.keys())}):\n")
    for name in sorted(tree.keys()):
        print(f"  {name}")

In [ ]:
# Show branches for a specific collection prefix

COLLECTION = "Muon"  # change to "Electron", "Jet", "Tau", "FatJet", "MET", etc.

if NANOAOD_PATH is None:
    print("NANOAOD_PATH not set — showing expected branches from this analysis instead.")
    coll_branches = df.loc[COLLECTION] if COLLECTION in df.index.get_level_values(0) else pd.DataFrame()
    print(coll_branches.to_string())
else:
    coll_keys = [k for k in tree.keys() if k.startswith(f"{COLLECTION}_")]
    print(f"{COLLECTION} branches ({len(coll_keys)}):")
    for k in coll_keys:
        dtype = tree[k].interpretation
        print(f"  {k:40s}  {dtype}")

In [ ]:
# Load a small batch of muon branches into awkward arrays (requires real file)

if NANOAOD_PATH is None:
    print("NANOAOD_PATH not set — skipping.")
else:
    import awkward as ak

    muon_branches = ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_tightId", "Muon_pfIsoId"]
    arrays = tree.arrays(muon_branches, entry_stop=1000, library="ak")

    print(f"Loaded {len(arrays['Muon_pt'])} events")
    print(f"Muon_pt type: {arrays['Muon_pt'].type}")
    print()

    # Count muons per event
    n_muons = ak.num(arrays["Muon_pt"])
    print(f"Muons per event — mean: {ak.mean(n_muons):.2f}, max: {ak.max(n_muons)}")

    # Select muons with pt > 20 and |eta| < 2.4
    mask = (arrays["Muon_pt"] > 20) & (abs(arrays["Muon_eta"]) < 2.4)
    selected_pts = arrays["Muon_pt"][mask]
    print(f"After pt>20 and |eta|<2.4: {ak.sum(ak.num(selected_pts))} muons in first 1000 events")

## NanoAOD v12 vs v15 — differences relevant to this analysis

| Feature | v12 (2022) | v15 (2024–2025) |
|---|---|---|
| MET branch name | `MET_pt`, `MET_phi` | `PFMET_pt`, `PFMET_phi` (priority) |
| Electron loose ID | `Electron_mvaIso_WP90` optional | `Electron_mvaIso_WP90` present |
| Muon loose ID | `Muon_looseId` optional | `Muon_looseId` typically present |
| Jet flavour | `Jet_hadronFlavour` or `Jet_partonFlavour` | same (hadronFlavour preferred) |
| DeepTau version | `Tau_idDeepTau2018v2p5VSjet` | same |
| Pileup branch | `Pileup_nTrueInt` | same |

The framework handles these differences in `darkbottomline/objects.py` via `if branch in events.fields` guards.

## Event selection summary

After loading branches, the analysis applies the following event-level preselection
(from `darkbottomline/selections.py` and the year config):

In [ ]:
selection = cfg.get("event_selection", {})
print("Event preselection cuts (2022 config):")
for key, val in selection.items():
    print(f"  {key}: {val}")

print()
print("Trigger paths (2022 config):")
for trig_type, paths in cfg.get("triggers", {}).items():
    if isinstance(paths, list):
        for p in paths:
            print(f"  [{trig_type}] {p}")
    else:
        print(f"  [{trig_type}] {paths}")